In [ ]:
pip install emoji

In [ ]:
pip install optuna

In [ ]:
# import les packages
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import cross_val_score,KFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, accuracy_score, roc_auc_score
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.pipeline import make_pipeline
import re
import requests
from bs4 import BeautifulSoup
import emoji
from joblib import Memory
import matplotlib.pyplot as plt
import spacy
import optuna
import json

In [ ]:
import warnings
warnings.filterwarnings("ignore",category=UserWarning)

# I. Preprocessing Data

In [ ]:
# Open the JSON file and return the data as a DATAFRAME
def read_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return pd.DataFrame(data).T

In [ ]:
df=read_json_file('EXIST2024_training.json')
df.head()

In [ ]:
def count(x):
    c=0
    for word in x:
        if word=='YES':
            c+=1
    return c
df['sexism']=df.labels_task1.apply(lambda x:count(x)) # number of annotators that identifies sexism
df['label']=df['sexism'].apply(lambda x: 1 if x>=3 else 0)
df=df[['tweet','label','lang']]

In [ ]:
df_1=read_json_file('EXIST2023_dev.json')
df_2=pd.read_csv('EXIST2021_training.tsv',delimiter='\t')
df_1['label']=df_1.labels_task1.apply(lambda x:count(x)).apply(lambda x: 1 if x>=3 else 0)
df_1=df_1[['tweet','label','lang']]
df_2=df_2[['text','task1','language']].rename( {'text':'tweet','task1':'label',
                                                'language':'lang'},axis=1)
df_2['label']=df_2['label'].apply(lambda x: 1 if x=='sexist' else 0)
df=pd.concat([df,df_1,df_2],ignore_index=True)
df

In [ ]:
df.describe()

In [ ]:
def contains_url_or_emoji(text):
    # Check if the text contains a URL
    url_pattern = r'http[s]?://\S+'  # URL pattern
    contains_url = bool(re.search(url_pattern, text))

    # Check if the text contains any emoji
    contains_emoji = emoji.emoji_count(text) > 0  # Using emoji.emoji_count to check for emojis
    return contains_url or contains_emoji  # Return True if either URL or emoji is found

# Function to count the number of tweets that contain a URL or emoji
def count_tweets_with_url_or_emoji(df, column_name):
    count = df[column_name].apply(contains_url_or_emoji).sum()
    return count

In [ ]:
count_tweets_with_url_or_emoji(df,'tweet')/len(df)

<blockquote style="border-left: 4px solid #8e44ad; padding-left: 15px; font-style: italic; color: #8e44ad; background-color: #f9f9f9; border-radius: 5px;">
35% of the data contain either emoji or url.

Now we will preprocess the tweet
</blockquote>


In [ ]:
def get_url_title(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        title = soup.title.string
        return title
    except:
        return "URL_ERROR"

# Function to clean and preprocess tweets
def process_url_tweet(text):
    # Replace URLs with their titles
    urls = re.findall(r'http[s]?://\S+', text)
    for url in urls:
        title = get_url_title(url)  # Fetch title for the URL
        if title is None or title == "URL_ERROR" or title == "No Title Found":
            text = text.replace(url, "")  # Remove the URL if title fetching failed
        else:

            text = text.replace(url, title)  # Replace URL with title

    return text


def preprocess_tweet(text):

    # 1. Remove URLs or Replace it by the Title
    #text = process_url_tweet(text)
    text = re.sub(r'http[s]?://\S+', '', text)

    # 2. Remove user tags (@username)
    text = re.sub(r'@\w+', '', text)

    # 3. Handle the emojis
    text=emoji.demojize(text)

    # 4. Remove numbers and standalone dates
    text = re.sub(r'\b\d{1,4}[-/]\d{1,2}[-/]\d{1,4}\b', '', text)  # dates like 2023-01-01
    text = re.sub(r'\b\d+\b', '', text)

    # 5. Remove inverted Spanish punctuation (¡ and ¿)
    text = re.sub(r'[¡¿]', '', text)

    # 6. Fix missing space after punctuation (e.g., "hello.This" → "hello. This")
    text = re.sub(r'([.!?])([^\s])', r'\1 \2', text)

    # 7. Normalize repeated punctuation: !!??!? → !!
    text = re.sub(r'([!?]){2,}', r'\1\1', text)
    text = re.sub(r'(!\?|\?!)+', '!?', text)

    # 8. Normalize repeated letters (e.g., soooo → soo)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 9. Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # 10. Strip leading/trailing spaces
    text = text.strip()

    # 11. Remove any slash followed by alphanumeric characters or symbols
    text = re.sub(r"\/", "", text)

    return text.strip() # Strip any leading/trailing spaces that may remain

In [ ]:
df['tweet']=df['tweet'].apply(lambda x:preprocess_tweet(x))
df.head()

In [ ]:
#split the english and spanish tweets
df_es=df[df['lang']=='es']
df_en=df[df['lang']=='en']
df_en.head()

In [ ]:
print("Proportion of English tweets",len(df_en)/len(df))
print("Proportion of Spanish tweets",len(df_es)/len(df))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(8, 8))
df_en['label'].value_counts(normalize=True).plot(kind='pie',autopct='%1.1f%%',labels=['Not Sexist','Sexist'],radius=0.8,ax=ax[0])
df_es['label'].value_counts(normalize=True).plot(kind='pie',autopct='%1.1f%%',labels=['Sexist','Not Sexist'],radius=0.8,ax=ax[1])
ax[0].set_title('Sexism in English Tweets', fontsize=12)
ax[1].set_title('Sexism in Spanish Tweets', fontsize=12)
plt.show()

In [ ]:
from wordcloud import WordCloud
from wordcloud import STOPWORDS

def plot_word(df):
    """
    Function to plot wordcloud
    """
    # Wordcloud with positive tweets
    positive_tweets = df['tweet'][df["label"] == 0] # no sexism
    stop_words = ["https", "co", "RT"] + list(STOPWORDS)
    positive_wordcloud = WordCloud(max_font_size=50, max_words=100, background_color="white", stopwords = stop_words).generate(str(positive_tweets))
    plt.figure()
    plt.title("Non sexist Tweets - Wordcloud")
    plt.imshow(positive_wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()

    # Wordcloud with negative tweets
    negative_tweets = df['tweet'][df["label"] == 1] # sexism
    stop_words = ["https", "co", "RT"] + list(STOPWORDS)
    negative_wordcloud = WordCloud(max_font_size=50, max_words=100, background_color="white", stopwords = stop_words).generate(str(negative_tweets))
    plt.figure()
    plt.title("Sexist Tweets - Wordcloud")
    plt.imshow(negative_wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()


In [ ]:
plot_word(df_en)
plot_word(df_es)

1. Machine Learning Models

In [ ]:
!python -m spacy download es_core_news_sm


In [ ]:
nlp_en = spacy.load("en_core_web_sm")
nlp_es = spacy.load("es_core_news_sm")

def lemmatize_tweet(text,lang):
    # Function to lemmatize the tweet text
    if lang=='en':
        doc = nlp_en(text)
        return " ".join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct])
    else:
        doc = nlp_es(text)
        return " ".join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct])


text_en=df_en['tweet'].apply(lambda x: lemmatize_tweet(x,'en'))
text_es=df_es['tweet'].apply(lambda x: lemmatize_tweet(x,'es'))

In [ ]:
X_train_en,X_test_en,Y_train_en,Y_test_en=train_test_split(text_en,df_en['label'],stratify=df_en['label'],
                                                           test_size=0.2,random_state=42)
X_train_es,X_test_es,Y_train_es,Y_test_es=train_test_split(text_es,df_es['label'],stratify=df_es['label'],
                                                           test_size=0.2,random_state=42)

In [ ]:
# Initialize the TF-IDF Vectorizer
def create_tfidf_vectorizer(text,lang):
    tfidf_vectorizer =TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(text)
    return tfidf_matrix.toarray(),tfidf_vectorizer

# Fit and transform the corpus
x_train_en,tfidf_en=create_tfidf_vectorizer(X_train_en,'en')
x_train_es,tfidf_es=create_tfidf_vectorizer(X_train_es,'es')
x_test_en=tfidf_en.transform(X_test_en).toarray()
x_test_es=tfidf_es.transform(X_test_es).toarray()

In [ ]:
print(x_train_en.shape,x_train_es.shape)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
X_pca_en = pca.fit_transform(x_train_en-x_train_en.mean(axis=0))

# Step 4: Explained variance ratio (percentage of variance explained by each principal component)
explained_variance_ratio = pca.explained_variance_ratio_


# Step 6: Choose how many components to keep (e.g., keep 95% of the variance)
cumulative_variance = np.cumsum(explained_variance_ratio)
n_components = np.where(cumulative_variance >= 0.8)[0][0] + 1  # Components to retain 95% variance
print(f"Number of components to keep for 80% variance: {n_components}")

pca = PCA(n_components=n_components)
X_pca_reduced = pca.fit_transform(x_train_en-x_train_en.mean(axis=0))

In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Objective function for Logistic Regression
def study_logregression(X_train,Y_train):
    # Get the best parameters and AUC score for each model
    def objective_logreg(trial):
        C = trial.suggest_loguniform('C', 0.1,2)
        class_weight = trial.suggest_categorical('class_weight', ['balanced'])
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

        model = LogisticRegression(C=C, solver='liblinear', class_weight=class_weight,
                                fit_intercept=True, penalty=penalty, max_iter=100, random_state=42)

        # Perform cross-validation and return the mean AUC score
        auc_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='roc_auc')
        return auc_scores.mean()
    study_logreg = optuna.create_study(direction='maximize')
    study_logreg.optimize(objective_logreg, n_trials=50)
    return study_logreg.best_params, study_logreg.best_value


# Objective function for Naive Bayes
def study_naive_bayes(X_train,Y_train):
    def objective_nb(trial):
        var_smoothing = trial.suggest_loguniform('var_smoothing', 1, 20)

        model = GaussianNB(var_smoothing=var_smoothing)

        # Perform cross-validation and return the mean AUC score
        auc_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='roc_auc')
        return auc_scores.mean()
    study_NB = optuna.create_study(direction='maximize')
    study_NB.optimize(objective_nb, n_trials=50)
    return study_NB.best_params, study_NB.best_value



# Objective function for XGBoost
def study_xgboost(X_train,Y_train):
    def objective_xgboost(trial):
        learning_rate = trial.suggest_float('learning_rate', 0.05, 0.2)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        subsample = trial.suggest_float('subsample', 0.7, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0)
        reg_alpha = trial.suggest_float("reg_alpha", 0, 1.0)
        reg_lambda = trial.suggest_float("reg_lambda", 0, 1.0)

        model = xgb.XGBClassifier(learning_rate=learning_rate, max_depth=max_depth, n_estimators=50, subsample=subsample,
                                colsample_bytree=colsample_bytree, use_label_encoder=False,
                                reg_alpha=reg_alpha, reg_lambda=reg_lambda, eval_metric="mlogloss", objective="binary:logistic")

        # Perform cross-validation and return the mean AUC score
        auc_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='roc_auc')
        return auc_scores.mean()
    study_xgb=optuna.create_study(direction='maximize')
    study_xgb.optimize(objective_xgboost, n_trials=50)
    return study_xgb.best_params, study_xgb.best_value

# Objective function for SVM
def study_svm(X_train,Y_train):
    def objective_svm(trial):
        # Suggest hyperparameters for SVM
        kernel = trial.suggest_categorical('kernel', ['linear', 'poly'])
        C = trial.suggest_loguniform('C', 1, 100)  # Regularization parameter (log scale)
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto']) if kernel != 'linear' else 'auto'

        # Correctly handle degree and coef0 for different kernels:
        degree = trial.suggest_int('degree', 2, 5) if kernel == 'poly' else 3  # Default value for other kernels
        coef0 = trial.suggest_float('coef0', 0.0, 10.0) if kernel in ['poly', 'sigmoid'] else 0.0

        model = SVC(C=C, kernel=kernel, gamma=gamma, degree=degree, coef0=coef0, probability=True, random_state=42)

        # Perform cross-validation and return the mean AUC score
        auc_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='roc_auc')
        return auc_scores.mean()
    study_svm=optuna.create_study(direction='maximize')
    study_svm.optimize(objective_svm, n_trials=25)
    return study_svm.best_params, study_svm.best_value

In [ ]:
study_logregression(x_train_en,Y_train_en)

In [ ]:
study_xgboost(X_pca_reduced,Y_train_en)

In [ ]:
study_svm(X_pca_reduced,Y_train_en)

In [ ]:
models={
    'Logistic Regression': LogisticRegression( C=0.75, solver='liblinear', class_weight='balanced',
                               fit_intercept=True, penalty='l1',
                              max_iter=100, random_state=42),

    'Naive Bayes': GaussianNB(var_smoothing=10),

    'XGBoost': xgb.XGBClassifier(max_depth=5,learning_rate=0.05, subsample = 0.8,
                colsample_bytree=0.8, reg_alpha=0.85, reg_lambda=0.25, n_estimators=50,
                use_label_encoder=False, eval_metric='mlogloss', objective='binary:logistic'),

    'SVM': SVC(C=7,kernel='poly',gamma='auto',degree=3,coef0=5,probability=True)
    }

def evaluate_models(X_train, Y_train, X_test, Y_test,model):
    """
    Function to evaluate the models
    """
    # Fit the model
    model.fit(X_train, Y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate accuracy and F1 score
    accuracy = accuracy_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
    return accuracy, f1

<blockquote style="border-left: 4px solid #8e44ad; padding-left: 15px; font-style: italic; color: #8e44ad; background-color: #f9f9f9; border-radius: 5px;">
Apply the models to the Test Dataset
</blockquote>


In [ ]:
for model in models.keys():
    accuracy, f1 = evaluate_models(x_train_en, Y_train_en, x_test_en, Y_test_en, models[model])
    print(f"{model} - Accuracy English: {accuracy:.4f}, F1 Score: {f1:.4f}")
    accuracy, f1 = evaluate_models(x_train_es, Y_train_es, x_test_es, Y_test_es, models[model])
    print(f"{model} - Accuracy Spanish: {accuracy:.4f}, F1 Score: {f1:.4f}")


## Transformers Based models:

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a dataset class
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])  # Don't move to device here
        return item

    def __len__(self):
        return len(self.labels)

def train_bert(X_train, Y_train, X_test, Y_test,lang):
    """
    Train the BERT model
    """
    if lang=='en':
      model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
      tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    else:
      tokenizer = BertTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')
      model = BertForSequenceClassification.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased', num_labels=2)

    # Tokenize input data
    train_encodings = tokenizer(X_train.tolist(), return_tensors="pt", truncation=True, padding=True)
    train_labels = torch.tensor(Y_train.tolist())
    test_encodings = tokenizer(X_test.tolist(), return_tensors="pt", truncation=True, padding=True)
    test_labels = torch.tensor(Y_test.tolist())

    # Create datasets
    train_dataset = TweetDataset(train_encodings, train_labels)
    test_dataset = TweetDataset(test_encodings, test_labels)

    # Move model to the device (GPU/CPU)
    model.to(device)

    # Define training arguments
    training_args = TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        save_strategy="epoch",
        output_dir="./results",
        eval_steps=100,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train()
    return trainer, test_dataset

def evaluate_bert(trainer, test_dataset):
    """
    Evaluate the BERT model
    """
    # Make predictions
    predictions = trainer.predict(test_dataset)

    # Extract logits and convert to predicted class (0 or 1)
    logits = predictions.predictions
    predicted_labels = np.argmax(logits, axis=1)

    # Get true labels
    true_labels = predictions.label_ids

    return predicted_labels, true_labels


# During training or evaluation, ensure that inputs are moved to the correct device
def collate_fn(batch):
    # Move tensors to device inside the collate_fn for batches during training/evaluation
    inputs = {key: torch.stack([x[key] for x in batch]).to(device) for key in batch[0].keys()}
    return inputs



In [ ]:
x_train_es,x_test_es,y_train_es,y_test_es=train_test_split(df_es['tweet'],df_es['label'],stratify=df_es['label'],
                                                           test_size=0.2,random_state=42)
x_train_en,x_test_en,y_train_en,y_test_en=train_test_split(df_en['tweet'],df_en['label'],stratify=df_en['label'],
                                                           test_size=0.2,random_state=42)

In [ ]:
trainer_en,test_dataset_en=train_bert(x_train_en,y_train_en,x_test_en,y_test_en,"en")
predicted_labels_en,true_labels_en=evaluate_bert(trainer_en,test_dataset_en)
print(classification_report(true_labels_en,predicted_labels_en,target_names=["non-sexist", "sexist"]))
print("f1_score English",f1_score(true_labels_en, predicted_labels_en))
print("accuracy English",accuracy_score(true_labels_en, predicted_labels_en))

In [ ]:
trainer_es,test_dataset_es=train_bert(x_train_es,y_train_es,x_test_es,y_test_es,"en")
predicted_labels_es,true_labels_es=evaluate_bert(trainer_es,test_dataset_es)
print(classification_report(true_labels_es,predicted_labels_es,target_names=["non-sexist", "sexist"]))
print("f1_score Spanish",f1_score(true_labels_es, predicted_labels_es))
print("accuracy Spanish",accuracy_score(true_labels_es, predicted_labels_es))

In [ ]:
trainer_es,test_dataset_es=train_bert(x_train_es,y_train_es,x_test_es,y_test_es,"es")
predicted_labels_es,true_labels_es=evaluate_bert(trainer_es,test_dataset_es)
print(classification_report(true_labels_es,predicted_labels_es,target_names=["non-sexist", "sexist"]))
print("f1_score Spanish",f1_score(true_labels_es, predicted_labels_es))
print("accuracy Spanish",accuracy_score(true_labels_es, predicted_labels_es))

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import EvalPrediction
from sklearn.metrics import precision_recall_fscore_support
from transformers import XLMRobertaTokenizer, XLMRobertaModel

# 1. Enhanced Model Definition
class XLMRobertaWithGeLU(nn.Module):
    def __init__(self, num_classes=2, hidden_dropout_prob=0.1):
        super().__init__()
        self.num_classes = num_classes

        # Load pretrained model with gradient checkpointing
        self.xlm_roberta = XLMRobertaModel.from_pretrained(
            "xlm-roberta-base",
        )

        # Enhanced classifier with LayerNorm
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.xlm_roberta.config.hidden_size),
            nn.Linear(self.xlm_roberta.config.hidden_size, 512),
            nn.GELU(),
            nn.Dropout(hidden_dropout_prob),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.xlm_roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # Weighted average of last 4 layers
        hidden_states = outputs.hidden_states[-4:]
        cls_embeddings = torch.stack([state[:, 0] for state in hidden_states])
        cls_embedding = torch.mean(cls_embeddings, dim=0)

        logits = self.classifier(cls_embedding)

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(
                weight=torch.tensor([1.0, 2.0]).to(labels.device) )# Class weighting
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))
            return {'loss': loss, 'logits': logits}
        return {'logits': logits}

# 2. Optimized Dataset Class
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = {
            k: torch.tensor(v)
            for k, v in encodings.items()
        }
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        return {
            **{k: v[idx] for k, v in self.encodings.items()},
            'labels': self.labels[idx]
        }

    def __len__(self):
        return len(self.labels)


# 4. Training Function with Improvements
def train_roberta(X_train, Y_train, X_test, Y_test, device):
    tokenizer = XLMRobertaTokenizer.from_pretrained(
        "xlm-roberta-base",
        do_lower_case=False  # Important for multilingual
    )

    # Dynamic padding and truncation
    train_encodings = tokenizer(
        X_train.tolist(),
        truncation=True,
        padding='longest',
        max_length=256
    )
    test_encodings = tokenizer(
        X_test.tolist(),
        truncation=True,
        padding='longest',
        max_length=256
    )

    train_dataset = TweetDataset(train_encodings, Y_train)
    test_dataset = TweetDataset(test_encodings, Y_test)

    model = XLMRobertaWithGeLU().to(device)

    # Enhanced Training Arguments
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=3,  # Realistic training duration
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=100,
        save_strategy="steps",

    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    # Start training with progress bar
    trainer.train()

    return trainer, test_dataset

In [ ]:
x_train_tot,x_test_tot,y_train_tot,y_test_tot=train_test_split(df['tweet'].values,df['label'].values,
                                                              stratify=df['label'],test_size=0.2,
                                                              random_state=42)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer_tot,test_dataset_tot=train_roberta(x_train_tot,y_train_tot,x_test_tot,y_test_tot,device)
predicted_labels_tot,true_labels_tot=evaluate_bert(trainer_tot,test_dataset_tot)
print(classification_report(true_labels_tot,predicted_labels_tot,target_names=["non-sexist", "sexist"]))
print("f1_score Macro",f1_score(true_labels_tot, predicted_labels_tot))
print("accuracy ",accuracy_score(true_labels_tot, predicted_labels_tot))

# Memes Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#df_memes=pd.read_json('EXIST2024_memes_training.json').T
drive_path='/content/drive/MyDrive/EXIST2021-2024_datasets/'
df_memes=pd.read_csv(drive_path+'df_memes.csv')[['image','text','label','image_base64']]
df_memes.head()

In [ ]:
from PIL import Image
from io import BytesIO
import base64
def base64_to_pil(base64_string):
    img_data = base64.b64decode(base64_string)
    return Image.open(BytesIO(img_data))

df_memes['image']=df_memes['image_base64'].apply(lambda x: base64_to_pil(x))
df_memes=df_memes[['image','text','label']]

In [ ]:
df_memes['text']=df_memes['text'].apply(lambda x:preprocess_tweet(x))
#df_memes['sexism']=df_memes.labels_task4.apply(lambda x:count(x))
#df_memes['label']=df_memes['sexism'].apply(lambda x: 1 if x>=3 else 0)
df_memes_train,df_memes_test=train_test_split(df_memes,test_size=0.2,random_state=42,
                                              stratify=df_memes['label'])


In [ ]:
df_memes_train['image'].iloc[0]

In [ ]:
import torch
import torch.nn as nn
from transformers import XLMRobertaTokenizer, XLMRobertaModel, Trainer, TrainingArguments
from torchvision import models, transforms
from torch.utils.data import Dataset

# Define the Dataset Class
class MemeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, image_transform):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.images = dataframe['image'].values
        self.texts = dataframe['text'].values
        self.labels = dataframe['label'].values

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Process the image
        image = self.images[idx]
        image = self.image_transform(image)

        # Process the text
        text = self.texts[idx]
        text_inputs = self.tokenizer(text, truncation=True, padding='max_length', max_length=256, return_tensors="pt")

        # Get the label
        label = torch.tensor(self.labels[idx])

        return {
            'pixel_values': image,  # Changed from 'image' to standard HF name
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'labels': label  # Changed to 'labels' (plural) for HF compatibility
        }

# Define Multimodal Model
class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Image branch (ResNet-50)
        self.resnet = models.resnet50(pretrained=True)
        self.resnet_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # Text branch (XLM-RoBERTa)
        self.roberta = XLMRobertaModel.from_pretrained("xlm-roberta-base")

        # Fusion layer
        self.classifier = nn.Sequential(
            nn.Linear(self.resnet_features + self.roberta.config.hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward(self, pixel_values=None, input_ids=None, attention_mask=None, labels=None):
        # Image features
        img_features = self.resnet(pixel_values)
        img_features = img_features.view(img_features.size(0), -1)

        # Text features
        text_features = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]  # CLS token

        # Combined features
        combined = torch.cat([img_features, text_features], dim=1)
        logits = self.classifier(combined)

        # Calculate loss if labels are provided
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1))
            return {'loss': loss, 'logits': logits}
        return {'logits': logits}

# Training function
def train_resberta(df_train, df_test, device):
    tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
    image_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = MemeDataset(df_train, tokenizer, image_transform)
    test_dataset = MemeDataset(df_test, tokenizer, image_transform)

    model = MultimodalModel().to(device)

    # Enhanced Training Arguments
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=100,
        save_strategy="steps",
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train()
    return trainer, test_dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer_meme,test_dataset_meme=train_resberta(df_memes_train,df_memes_test,device)
predicted_labels,true_labels=evaluate_bert(trainer_meme,test_dataset_meme)
print(classification_report(true_labels,predicted_labels,target_names=["non-sexist", "sexist"]))
print("f1_score Macro",f1_score(true_labels, predicted_labels))
print("accuracy ",accuracy_score(true_labels, predicted_labels))

In [ ]:
# Method 1: Clear all user-defined variables
for name in dir():
    if not name.startswith('_'):  # Skip built-ins
        del globals()[name]

# Method 2: Reset the namespace (in IPython/Jupyter)
%reset -f  # Force reset without confirmation